```
Problem:
--------
Implement an API rate limiter allowing 5 accepted requests
per customer within a rolling 10-second window.

Concepts:
---------
- State management
- Hash tables
- Sliding window
- Queue (deque)
- Event-driven processing

Complexity:
-----------
Time: O(n)
Space: O(k)

Takeaways:
----------
A deque naturally models a sliding window because
new events append to the rear while expired events
leave from the front.
```

In [16]:
from collections import deque


def process_requests(requests):
    """
    Process API requests using a per-customer sliding window rate limit.

    Each customer is allowed up to 5 accepted requests within any 10-second
    window. Requests are processed in chronological order. For each request,
    the function determines whether the request should be accepted or rejected
    based on that customer's accepted request history within the current
    10-second window.

    Parameters:
        requests (list[tuple[str, int]]): A list of request tuples. Each tuple
        contains a customer ID and a timestamp in seconds.

        Example:
            [
                ("customer_a", 1),
                ("customer_a", 2),
                ("customer_b", 3),
            ]

    Returns:
        list[tuple[str, int, str]]: A list of tuples containing the customer ID,
        timestamp, and request status. The status is either "accepted" or
        "rejected".

        Example:
            [
                ("customer_a", 1, "accepted"),
                ("customer_a", 2, "accepted"),
                ("customer_b", 3, "accepted"),
            ]

    Assumptions:
        1. Requests are already ordered by timestamp from earliest to latest.
        2. Each request tuple contains exactly two values:
           (customer_id, timestamp).
        3. customer_id is a hashable value, such as a string.
        4. timestamp is an integer or numeric value representing seconds.
        5. Each customer has an independent rate limit.
        6. Rejected requests do not count toward the customer's rate limit.
        7. The rate limit is 5 accepted requests per 10-second window.
        8. A timestamp expires from the current window when:
           current_timestamp - old_timestamp >= 10.
           For example, at timestamp 12, a request from timestamp 2 has expired.
        9. A request is accepted only when the customer has fewer than 5
           accepted requests remaining in the current 10-second window.
        10. The input list may be empty. If it is empty, the function returns
            an empty list.
        11. A dictionary is used to maintain per-customer state because it
            provides efficient lookup by customer ID.
        12. A deque is used for each customer's accepted request timestamps
            because old timestamps can be efficiently removed from the front
            of the queue.
    """

    limiter = {}
    results = []
    for customer_id, timestamp in requests:  # parse the tuple
        if customer_id not in limiter:
            # add customer to dictionary
            limiter[customer_id] = deque()
            print(limiter)
            print(deque)
        customer_requests = limiter[customer_id]

        # remove expired timestamps
        while customer_requests and timestamp - customer_requests[0] >= 10:
            customer_requests.popleft()

        if len(customer_requests) < 5:
            customer_requests.append(timestamp)
            results.append((customer_id, timestamp, "accepted"))
        else:
            results.append((customer_id, timestamp, "rejected"))

    return results


In [17]:
requests = [
    ("customer_a", 1),
    ("customer_a", 2),
    ("customer_b", 3),
    ("customer_a", 4),
    ("customer_a", 6),
    ("customer_a", 8),
    ("customer_a", 9),
    ("customer_b", 11),
    ("customer_a", 12),
    ("customer_a", 13),
    ("customer_a", 14),
    ("customer_b", 15),
    ("customer_a", 16),
    ("customer_a", 17),
    ("customer_a", 18),
    ("customer_a", 19),
    ("customer_b", 20),
    ("customer_a", 22),
]

In [18]:
process_requests(requests)

{'customer_a': deque([])}
<class 'collections.deque'>
{'customer_a': deque([1, 2]), 'customer_b': deque([])}
<class 'collections.deque'>


[('customer_a', 1, 'accepted'),
 ('customer_a', 2, 'accepted'),
 ('customer_b', 3, 'accepted'),
 ('customer_a', 4, 'accepted'),
 ('customer_a', 6, 'accepted'),
 ('customer_a', 8, 'accepted'),
 ('customer_a', 9, 'rejected'),
 ('customer_b', 11, 'accepted'),
 ('customer_a', 12, 'accepted'),
 ('customer_a', 13, 'accepted'),
 ('customer_a', 14, 'accepted'),
 ('customer_b', 15, 'accepted'),
 ('customer_a', 16, 'accepted'),
 ('customer_a', 17, 'rejected'),
 ('customer_a', 18, 'accepted'),
 ('customer_a', 19, 'rejected'),
 ('customer_b', 20, 'accepted'),
 ('customer_a', 22, 'accepted')]